In [23]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Configuration
DATA_DIR = Path("../data/raw")
ELO_K = 20
ELO_INIT = 1500
ELO_HCA = 100

# Global-like state to mirror the ADK example
DATA = {}
ELO = {}
MODEL = None

print("Modeling notebook: config ready")

Modeling notebook: config ready


In [24]:
# TOOL 1: Load competition data (no ADK, just a function)

def load_competition_data() -> dict:
    DATA['m_teams'] = pd.read_csv(DATA_DIR / 'MTeams.csv')
    DATA['w_teams'] = pd.read_csv(DATA_DIR / 'WTeams.csv')
    DATA['m_regular'] = pd.read_csv(DATA_DIR / 'MRegularSeasonCompactResults.csv')
    DATA['w_regular'] = pd.read_csv(DATA_DIR / 'WRegularSeasonCompactResults.csv')
    DATA['m_tourney'] = pd.read_csv(DATA_DIR / 'MNCAATourneyCompactResults.csv')
    DATA['w_tourney'] = pd.read_csv(DATA_DIR / 'WNCAATourneyCompactResults.csv')
    DATA['m_seeds'] = pd.read_csv(DATA_DIR / 'MNCAATourneySeeds.csv')
    DATA['w_seeds'] = pd.read_csv(DATA_DIR / 'WNCAATourneySeeds.csv')
    DATA['sample_sub'] = pd.read_csv(DATA_DIR / 'SampleSubmissionStage1.csv')

    return {
        'status': 'success',
        'seasons': f"{DATA['m_regular']['Season'].min()}-{DATA['m_regular']['Season'].max()}",
        'mens_teams': len(DATA['m_teams']),
        'womens_teams': len(DATA['w_teams']),
        'regular_season_games': len(DATA['m_regular']) + len(DATA['w_regular']),
        'tourney_games': len(DATA['m_tourney']) + len(DATA['w_tourney']),
        'submission_rows': len(DATA['sample_sub']),
    }

summary = load_competition_data()
print("Data loaded:")
print(f"- Men's teams: {summary['mens_teams']}")
print(f"- Women's teams: {summary['womens_teams']}")
print(f"- Regular season games: {summary['regular_season_games']}")
print(f"- Tournament games: {summary['tourney_games']}")
print(f"- Seasons: {summary['seasons']}")
print("Data is ready for the next stage.")

Data loaded:
- Men's teams: 381
- Women's teams: 379
- Regular season games: 337648
- Tournament games: 4302
- Seasons: 1985-2026
Data is ready for the next stage.


In [25]:
# TOOL 2: Compute Elo ratings (same logic as in 02_feature_engineering)

def _run_elo(regular_df, tourney_df):
    elo = {}
    season_elos = {}
    all_games = pd.concat([regular_df, tourney_df]).sort_values(['Season', 'DayNum'])
    prev_season = None

    for _, row in all_games.iterrows():
        season = row['Season']
        if season != prev_season and prev_season is not None:
            for tid, r in elo.items():
                season_elos[(prev_season, tid)] = r
            elo = {tid: 0.75 * r + 0.25 * ELO_INIT for tid, r in elo.items()}
        prev_season = season

        w_id, l_id = row['WTeamID'], row['LTeamID']
        w_elo = elo.get(w_id, ELO_INIT)
        l_elo = elo.get(l_id, ELO_INIT)

        w_loc = row.get('WLoc', 'N')
        w_adj = w_elo + (ELO_HCA if w_loc == 'H' else (-ELO_HCA if w_loc == 'A' else 0))

        exp_w = 1.0 / (1.0 + 10 ** ((l_elo - w_adj) / 400.0))
        elo[w_id] = w_elo + ELO_K * (1.0 - exp_w)
        elo[l_id] = l_elo + ELO_K * (0.0 - (1.0 - exp_w))

    if prev_season is not None:
        for tid, r in elo.items():
            season_elos[(prev_season, tid)] = r

    return season_elos


def compute_elo_ratings() -> dict:
    m_elos = _run_elo(DATA['m_regular'], DATA['m_tourney'])
    w_elos = _run_elo(DATA['w_regular'], DATA['w_tourney'])
    ELO.update(m_elos)
    ELO.update(w_elos)

    m_names = dict(zip(DATA['m_teams']['TeamID'], DATA['m_teams']['TeamName']))
    w_names = dict(zip(DATA['w_teams']['TeamID'], DATA['w_teams']['TeamName']))
    latest_m = max(s for s, _ in m_elos.keys())
    latest_w = max(s for s, _ in w_elos.keys())
    top_m = sorted([(tid, r) for (s, tid), r in m_elos.items() if s == latest_m], key=lambda x: -x[1])[:5]
    top_w = sorted([(tid, r) for (s, tid), r in w_elos.items() if s == latest_w], key=lambda x: -x[1])[:5]

    return {
        'status': 'success',
        'total_ratings': len(ELO),
        'top_mens': [f"{m_names.get(t, t)}: {r:.0f}" for t, r in top_m],
        'top_womens': [f"{w_names.get(t, t)}: {r:.0f}" for t, r in top_w],
        'latest_m': latest_m,
        'latest_w': latest_w,
    }

feat_summary = compute_elo_ratings()
print("Elo ratings computed.\n")
print("Top men's teams:")
for line in feat_summary['top_mens']:
    print(f"- {line}")
print("\nTop women's teams:")
for line in feat_summary['top_womens']:
    print(f"- {line}")
print(f"\nFeatures are ready for model training (through {feat_summary['latest_m']} / {feat_summary['latest_w']}).")

Elo ratings computed.

Top men's teams:
- Houston: 1821
- Duke: 1819
- Arizona: 1789
- Connecticut: 1785
- Gonzaga: 1756

Top women's teams:
- Connecticut: 1913
- South Carolina: 1893
- UCLA: 1869
- Texas: 1858
- LSU: 1807

Features are ready for model training (through 2026 / 2026).


In [26]:
# TOOL 3: Train prediction model (logistic regression on Elo + seed diff)

def _parse_seed(seed_str: str) -> int:
    return int(seed_str[1:3])


def train_prediction_model() -> dict:
    global MODEL

    seed_map = {}
    for df in [DATA['m_seeds'], DATA['w_seeds']]:
        for _, row in df.iterrows():
            seed_map[(row['Season'], row['TeamID'])] = _parse_seed(row['Seed'])

    X, y = [], []
    for t_df in [DATA['m_tourney'], DATA['w_tourney']]:
        for _, row in t_df.iterrows():
            season = row['Season']
            if season < 2003:
                continue

            w_id, l_id = row['WTeamID'], row['LTeamID']
            w_elo = ELO.get((season - 1, w_id), ELO_INIT)
            l_elo = ELO.get((season - 1, l_id), ELO_INIT)
            w_seed = seed_map.get((season, w_id), 8)
            l_seed = seed_map.get((season, l_id), 8)

            if w_id < l_id:
                X.append([w_elo - l_elo, l_seed - w_seed])
                y.append(1)
            else:
                X.append([l_elo - w_elo, w_seed - l_seed])
                y.append(0)

    X = np.array(X)
    y = np.array(y)

    MODEL = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000)
    MODEL.fit(X, y)

    cv_scores = cross_val_score(
        LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000),
        X,
        y,
        scoring='neg_brier_score',
        cv=5,
    )
    brier = -cv_scores.mean()

    return {
        'status': 'success',
        'training_games': len(y),
        'win_rate_label1': float(y.mean()),
        'cv_brier_score': float(brier),
        'coefficients': {
            'elo_diff': float(MODEL.coef_[0][0]),
            'seed_diff': float(MODEL.coef_[0][1]),
            'intercept': float(MODEL.intercept_[0]),
        },
    }

model_summary = train_prediction_model()
print(f"The logistic regression model was trained on {model_summary['training_games']} games "
      f"with a cross-validation Brier score of {model_summary['cv_brier_score']:.4f}.")
print("\nModel coefficients:")
print(f"- Elo difference: {model_summary['coefficients']['elo_diff']:.6f}")
print(f"- Seed difference: {model_summary['coefficients']['seed_diff']:.6f}")
print(f"- Intercept: {model_summary['coefficients']['intercept']:.6f}")

if abs(model_summary['coefficients']['seed_diff']) > abs(model_summary['coefficients']['elo_diff']):
    print("\nSeed difference matters more than Elo difference in this model.")
else:
    print("\nElo difference matters more than seed difference in this model.")

The logistic regression model was trained on 2851 games with a cross-validation Brier score of 0.1697.

Model coefficients:
- Elo difference: 0.003135
- Seed difference: 0.165367
- Intercept: 0.039667

Seed difference matters more than Elo difference in this model.


In [ ]:
# TOOL 4: Generate submission file (local path version)

def generate_submission(output_path: str = "../submission.csv") -> dict:
    sub = DATA['sample_sub'].copy()

    seed_map = {}
    for df in [DATA['m_seeds'], DATA['w_seeds']]:
        for _, row in df.iterrows():
            seed_map[(row['Season'], row['TeamID'])] = _parse_seed(row['Seed'])

    preds = []
    for _, row in sub.iterrows():
        season = int(row['ID'].split('_')[0])
        t1 = int(row['ID'].split('_')[1])
        t2 = int(row['ID'].split('_')[2])

        latest_season = season - 1

        e1 = ELO.get((latest_season, t1), ELO_INIT)
        e2 = ELO.get((latest_season, t2), ELO_INIT)
        s1 = seed_map.get((season, t1), 8)
        s2 = seed_map.get((season, t2), 8)

        features = np.array([[e1 - e2, s2 - s1]])
        prob = MODEL.predict_proba(features)[0][1]
        prob = float(np.clip(prob, 0.01, 0.99))
        preds.append(prob)

    sub['Pred'] = preds
    sub.to_csv(output_path, index=False)

    return {
        'status': 'success',
        'num_predictions': len(preds),
        'mean_pred': float(np.mean(preds)),
        'std_pred': float(np.std(preds)),
        'output_path': output_path,
    }

sub_summary = generate_submission()
print(f"The submission file was saved to '{sub_summary['output_path']}' with "
      f"{sub_summary['num_predictions']} predictions.")
print(f"Mean predicted probability: {sub_summary['mean_pred']:.4f}")
print(f"Std of predicted probabilities: {sub_summary['std_pred']:.4f}")



The submission file was saved to '../submission.csv' with 519144 predictions.
Mean predicted probability: 0.5052
Std of predicted probabilities: 0.1682

Ideas to improve the model:
1. Incorporate more features (efficiency stats, Massey Ordinals, recency weighting).
2. Try more flexible models (XGBoost, LightGBM, ensembles).
3. Add interaction terms and separate models for men and women.


In [28]:
%reload_ext autoreload
%autoreload 2
%pip install xgboost
import sys
from pathlib import Path
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score


sys.path.append(str(Path("../src")))

from features import (
    normalize_games,
    build_team_season_stats,
    build_matchup_dataset,add_seed_features,compute_elo_ratings
)

print("importing done!")

Note: you may need to restart the kernel to use updated packages.
importing done!


In [29]:
m_reg = pd.read_csv("../data/raw/MRegularSeasonCompactResults.csv")
w_reg = pd.read_csv("../data/raw/WRegularSeasonCompactResults.csv")
m_tour = pd.read_csv("../data/raw/MNCAATourneyCompactResults.csv")
w_tour = pd.read_csv("../data/raw/WNCAATourneyCompactResults.csv")

m_reg_norm = normalize_games(m_reg, "M")
w_reg_norm = normalize_games(w_reg, "W")
m_tour_norm = normalize_games(m_tour,"M")
w_tour_norm = normalize_games(w_tour,"W")

regular_season_games = pd.concat(
    [m_reg_norm],
    ignore_index=True
)
team_stats_reg = build_team_season_stats(regular_season_games)

tournament_games = pd.concat(
    [m_tour_norm],
    ignore_index=True
)


print("Regular season games:", len(regular_season_games))
print("Tournament games:", len(tournament_games))

Regular season games: 196823
Tournament games: 2585


In [30]:
tourney_matchups = build_matchup_dataset(
    tournament_games,
    team_stats_reg
)

print("Tournament matchup rows:", len(tourney_matchups))
tourney_matchups.head()

Tournament matchup rows: 2585


,Season,Team1,Team2,WinPctDiff,AvgPDDiff,Team1Win
0,1985,1116,1234,-0.030303,-8.377990,1
1,1985,1120,1345,-0.059310,-5.259615,1
2,1985,1207,1250,0.546616,20.939474,1
3,1985,1229,1425,0.062169,2.094474,1
4,1985,1242,1325,0.025926,2.217370,1


In [31]:
print(tourney_matchups[["WinPctDiff", "AvgPDDiff"]].corr())
print("Correlation with outcome:")
print(tourney_matchups.corr(numeric_only=True)["Team1Win"])

            WinPctDiff  AvgPDDiff
WinPctDiff    1.000000   0.732907
AvgPDDiff     0.732907   1.000000
Correlation with outcome:
Season       -0.010345
Team1         0.021381
Team2         0.073744
WinPctDiff    0.323965
AvgPDDiff     0.356897
Team1Win      1.000000
Name: Team1Win, dtype: float64


In [32]:
train = tourney_matchups[tourney_matchups["Season"] <= 2021]
val = tourney_matchups[tourney_matchups["Season"].isin([2022, 2023])]
test = tourney_matchups[tourney_matchups["Season"] == 2024]

print("Train:", len(train))
print("Val:", len(val))
print("Test:", len(test))

Train: 2317
Val: 134
Test: 67


In [33]:
features = ["WinPctDiff", "AvgPDDiff"]

X_train = train[features]
y_train = train["Team1Win"]

X_val = val[features]
y_val = val["Team1Win"]

X_test = test[features]
y_test = test["Team1Win"]

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score

model = LogisticRegression()
model.fit(X_train, y_train)

val_preds = model.predict_proba(X_val)[:, 1]

print("Validation Log Loss:", log_loss(y_val, val_preds))
print("Validation Accuracy:", accuracy_score(y_val, model.predict(X_val)))

Validation Log Loss: 0.6846634107094473
Validation Accuracy: 0.5970149253731343


In [34]:
def add_nonlinear_features(df):
    df = df.copy()
    
    df["WinPctDiff_sq"] = df["WinPctDiff"] ** 2
    df["AvgPDDiff_sq"] = df["AvgPDDiff"] ** 2
    
    df["Abs_WinPctDiff"] = df["WinPctDiff"].abs()
    df["Abs_AvgPDDiff"] = df["AvgPDDiff"].abs()
    
    df["Interaction"] = df["WinPctDiff"] * df["AvgPDDiff"]
    
    return df

In [35]:
train = tourney_matchups[tourney_matchups["Season"] < 2024]
val   = tourney_matchups[tourney_matchups["Season"] == 2024]

train_nl = add_nonlinear_features(train)
val_nl   = add_nonlinear_features(val)

print(train_nl.head())

   Season  Team1  Team2  WinPctDiff  AvgPDDiff  Team1Win  WinPctDiff_sq  \
0    1985   1116   1234   -0.030303  -8.377990         1       0.000918   
1    1985   1120   1345   -0.059310  -5.259615         1       0.003518   
2    1985   1207   1250    0.546616  20.939474         1       0.298789   
3    1985   1229   1425    0.062169   2.094474         1       0.003865   
4    1985   1242   1325    0.025926   2.217370         1       0.000672   

   AvgPDDiff_sq  Abs_WinPctDiff  Abs_AvgPDDiff  Interaction  
0     70.190724        0.030303       8.377990     0.253878  
1     27.663554        0.059310       5.259615     0.311950  
2    438.461558        0.546616      20.939474    11.445843  
3      4.386822        0.062169       2.094474     0.130212  
4      4.916730        0.025926       2.217370     0.057487  


In [36]:
#Non-Linear Model
features_nl = [
    "WinPctDiff",
    "AvgPDDiff",
    "WinPctDiff_sq",
    "AvgPDDiff_sq",
    "Abs_WinPctDiff",
    "Abs_AvgPDDiff",
    "Interaction"
]

X_train = train_nl[features_nl]
y_train = train_nl["Team1Win"]

X_val = val_nl[features_nl]
y_val = val_nl["Team1Win"]

model_nl = LogisticRegression(max_iter=1000)
model_nl.fit(X_train, y_train)

val_preds_nl = model_nl.predict_proba(X_val)[:, 1]

print("Nonlinear Validation Log Loss:", log_loss(y_val, val_preds_nl))
print("Nonlinear Validation Accuracy:", accuracy_score(y_val, model_nl.predict(X_val)))

Nonlinear Validation Log Loss: 0.6369914358824156
Nonlinear Validation Accuracy: 0.582089552238806


In [37]:
# ADD SEEDS
m_seeds = pd.read_csv("../data/raw/MNCAATourneySeeds.csv")
w_seeds = pd.read_csv("../data/raw/WNCAATourneySeeds.csv")

seeds = pd.concat([m_seeds], ignore_index=True)

seeds["SeedNum"] = seeds["Seed"].str.extract(r"(\d+)").astype(int)

print(seeds.head())
print(seeds["SeedNum"].describe())

   Season Seed  TeamID  SeedNum
0    1985  W01    1207        1
1    1985  W02    1210        2
2    1985  W03    1228        3
3    1985  W04    1260        4
4    1985  W05    1374        5
count    2626.000000
mean        8.637852
std         4.648260
min         1.000000
25%         5.000000
50%         9.000000
75%        13.000000
max        16.000000
Name: SeedNum, dtype: float64


In [38]:
tourney_matchups = build_matchup_dataset(tournament_games, team_stats_reg)
tourney_matchups = add_nonlinear_features(tourney_matchups)
tourney_matchups = add_seed_features(tourney_matchups, seeds)

print(tourney_matchups[["Team1Seed", "Team2Seed", "SeedDiff"]].head())
print(tourney_matchups.isna().sum())

   Team1Seed  Team2Seed  SeedDiff
0          9          8        -1
1         11          6        -5
2          1         16        15
3          9          8        -1
4          3         14        11
Season            0
Team1             0
Team2             0
WinPctDiff        0
AvgPDDiff         0
Team1Win          0
WinPctDiff_sq     0
AvgPDDiff_sq      0
Abs_WinPctDiff    0
Abs_AvgPDDiff     0
Interaction       0
Team1Seed         0
Team2Seed         0
SeedDiff          0
dtype: int64


In [39]:
#Seed Features
features_with_seeds = features_nl + ["SeedDiff"]

train_t = tourney_matchups[tourney_matchups["Season"] < 2024]
val_t   = tourney_matchups[tourney_matchups["Season"] == 2024]

X_train = train_t[features_with_seeds]
y_train = train_t["Team1Win"]

X_val = val_t[features_with_seeds]
y_val = val_t["Team1Win"]

In [40]:
#MODEL TRAINED W SEEDS
model_seed = LogisticRegression(max_iter=1000)
model_seed.fit(X_train, y_train)

val_preds_seed = model_seed.predict_proba(X_val)[:, 1]

print("Seed Model Log Loss:", log_loss(y_val, val_preds_seed))
print("Seed Model Accuracy:", accuracy_score(y_val, model_seed.predict(X_val)))

Seed Model Log Loss: 0.5778028293004288
Seed Model Accuracy: 0.7014925373134329


In [41]:
elo_df = compute_elo_ratings(m_reg, k=20)

print(elo_df.head())

   Season  TeamID          Elo
0    1985    1228  1616.967021
1    1985    1328  1661.592209
2    1985    1106  1466.251701
3    1985    1354  1449.221133
4    1985    1112  1569.203029


In [42]:
# Merge Team1 Elo
tourney_df = tourney_matchups.merge(
    elo_df.rename(columns={"TeamID": "Team1", "Elo": "Elo1"}),
    on=["Season", "Team1"],
    how="left"
)

# Merge Team2 Elo 
tourney_df = tourney_df.merge(
    elo_df.rename(columns={"TeamID": "Team2", "Elo": "Elo2"}),
    on=["Season", "Team2"],
    how="left"
)

# Now this will exist
tourney_df["EloDiff"] = tourney_df["Elo1"] - tourney_df["Elo2"]

print(tourney_df[["Elo1", "Elo2"]].isna().sum())

Elo1    0
Elo2    0
dtype: int64


In [43]:
# ELO Features

features_with_elo = features_with_seeds + ["EloDiff"]

train = tourney_df[tourney_df["Season"] < 2024]
val   = tourney_df[tourney_df["Season"] == 2024]

X_train = train[features_with_elo]
y_train = train["Team1Win"]

X_val = val[features_with_elo]
y_val = val["Team1Win"]

model = LogisticRegression()
model.fit(X_train, y_train)

val_preds = model.predict_proba(X_val)[:, 1]

print("Validation Log Loss:", log_loss(y_val, val_preds))
print("Validation Accuracy:", accuracy_score(y_val, model.predict(X_val)))


Validation Log Loss: 0.5793960949573306
Validation Accuracy: 0.7014925373134329


/Users/dhanush/Documents/march_mania_2026/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [44]:
features_xgb = [
    "EloDiff",
    "SeedDiff",
    "WinPctDiff",
    "AvgPDDiff",
    "WinPctDiff_sq",
    "AvgPDDiff_sq",
    "Abs_WinPctDiff",
    "Abs_AvgPDDiff",
    "Interaction",
]

In [45]:
from xgboost import XGBClassifier
from sklearn.metrics import brier_score_loss

# Train/validation split for XGBoost using Elo + other features
train_xgb = tourney_df[tourney_df["Season"] < 2024]
val_xgb   = tourney_df[tourney_df["Season"] == 2024]

X_train = train_xgb[features_xgb]
y_train = train_xgb["Team1Win"]

X_val = val_xgb[features_xgb]
y_val = val_xgb["Team1Win"]

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
)

xgb.fit(X_train, y_train)

val_preds_xgb = xgb.predict_proba(X_val)[:, 1]

print("XGB Validation Log Loss:", log_loss(y_val, val_preds_xgb))
print("XGB Validation Accuracy:", accuracy_score(y_val, (val_preds_xgb > 0.5).astype(int)))
print("XGB Validation Brier Score:", brier_score_loss(y_val, val_preds_xgb))

XGB Validation Log Loss: 0.6256390887195463
XGB Validation Accuracy: 0.6268656716417911
XGB Validation Brier Score: 0.2218354344367981


In [46]:
feat_imp = pd.Series(xgb.feature_importances_, index=features_xgb)
print(feat_imp.sort_values(ascending=False))

SeedDiff          0.411266
AvgPDDiff         0.086879
EloDiff           0.084433
Abs_WinPctDiff    0.078778
WinPctDiff        0.073108
WinPctDiff_sq     0.072092
Interaction       0.069065
AvgPDDiff_sq      0.066581
Abs_AvgPDDiff     0.057798
dtype: float32
